In [ ]:
import torch
from torch.utils.data import DataLoader

from entrega_3.utils.art_dataset import prepare_data_splits, ArtDataset, plot_art_grid
from entrega_3.utils.gan import CondGenerator, ProjectionCritic, train_wgan_gp

In [ ]:
from torchvision import transforms

gan_transforms = transforms.Compose([
    transforms.Resize((128, 128)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomVerticalFlip(p=0.5),

    transforms.RandomChoice([
        transforms.RandomRotation((0, 0)),
        transforms.RandomRotation((90, 90)),
        transforms.RandomRotation((180, 180)),
        transforms.RandomRotation((270, 270))
    ]),

    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),

    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [ ]:
ROOT_DIRECTORY = "../../data/emoart-cured/final-images"
MACRO_CLASSES=['color-field-painting']

train_list, _, _ = prepare_data_splits(ROOT_DIRECTORY, MACRO_CLASSES)

print(f"Total Train: {len(train_list)}")

In [ ]:
train_dataset = ArtDataset(train_list, transform=gan_transforms)

# Take a sample
plot_art_grid(train_dataset, MACRO_CLASSES)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)

In [ ]:
# Instanciamos todo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gen = CondGenerator(z_dim=100, num_classes=1).to(device)
crit = ProjectionCritic(num_classes=1).to(device)

In [ ]:
# Entrenar
history = train_wgan_gp(gen, crit, train_loader, device, epochs=600, classes=MACRO_CLASSES)